# NeXo v3.0 · Notebook 10 — Granger Causality Feature Gate

> **CRISP-DM Phase 4 — Modeling** (temporal causality / feature selection).
> **Input:** `curated/subscribers.parquet` + `processed/oss_aggregates.parquet` (from nb 01).
> **Output:** `granger_feature_gate.json` — (cause, effect, median_p, significant) edges.
> **Consumed by:** `services/api-gateway` `/granger-causality/lead-time` endpoint + dashboard Convergence panel.

## Pipeline
```
subscribers + oss_aggregates → area×month panel → Granger F-test per (cause→effect) pair
            → median p across areas → significance gate → granger_feature_gate.json
```

## Why Granger, not just correlation
Correlation says *X moves with Y*. **Granger causality** asks: *do past values of X improve the
prediction of Y beyond Y's own past?* i.e. *X moves BEFORE Y, with predictive power*.
Defense claim — "network anomalies precede customer complaints" — is a **directional, temporal**
statement → exactly what the Granger F-test measures (it is predictive precedence, not true causation).

## Reference
Granger, C.W.J. (1969). *Investigating Causal Relations by Econometric Models and Cross-spectral
Methods.* Econometrica 37(3), 424–438.

## What this notebook does NOT do
- Does NOT prove philosophical causation — Granger = predictive precedence only.
- Does NOT train an ML model — it is a statistical feature-selection gate.
- Does NOT rename the output JSON keys (`lag_max_months`, `significance_threshold`, `edges`,
  `significant_edges`) — they are an API contract consumed by api-gateway.

## Known methodology gaps (documented, not yet applied — see reports/10_findings.md)
ADF/KPSS stationarity gate · differencing · BIC lag selection · Benjamini–Hochberg FDR ·
reverse-Granger sanity test. These strengthen rigor but change the gate's numbers → applied
in a measured follow-up after the defense freeze.


## 1 · Imports

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
from IPython.display import display

import io, json, os
from pathlib import Path
import boto3, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from botocore.client import Config
from statsmodels.tsa.stattools import grangercausalitytests
sns.set_theme(style='whitegrid')
s3 = boto3.client('s3', endpoint_url=os.environ.get('S3_ENDPOINT','http://localhost:9000'),
                  aws_access_key_id='minio', aws_secret_access_key='minio_pw',
                  config=Config(signature_version='s3v4'))

In [ ]:
# ── Parameters (papermill-overridable) ──────────────────────────────────────
# This cell is tagged `parameters`. Values below feed the Granger gate.
# NOTE: the OUTPUT JSON keys (lag_max_months, significance_threshold, edges,
#   significant_edges) are an API contract and must NOT be renamed — only their
#   VALUES are parameterized here.
import os, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

GRANGER_MAX_LAG  = 4          # test lags 1..4 months (max precedence horizon)
GRANGER_ALPHA    = 0.05       # significance threshold on the F-test p-value
MIN_PANEL_OBS    = 5          # skip an area with fewer than this many monthly points
GRANGER_FTEST    = 'ssr_ftest'  # statsmodels test key (sum-of-squared-residuals F-test)

# Causes (OSS KPIs) and effects (BSS experience metrics) to test
GRANGER_CAUSES   = ['integ', 'cdr', 'tput']
GRANGER_EFFECTS  = ['cem_target', 'rat_gap', 'churn']

GATE_FNAME       = 'granger_feature_gate.json'  # IMMUTABLE output name (api-gateway reads it)

print(f'SEED={SEED} | maxlag={GRANGER_MAX_LAG} | alpha={GRANGER_ALPHA} | '
      f'min_obs={MIN_PANEL_OBS} | causes={GRANGER_CAUSES} effects={GRANGER_EFFECTS}')


## 2 · Load curated subscribers + processed OSS aggregates

In [ ]:
subs = pd.read_parquet(io.BytesIO(s3.get_object(Bucket='curated', Key='subscribers.parquet')['Body'].read()))
agg = pd.read_parquet(io.BytesIO(s3.get_object(Bucket='processed', Key='oss_aggregates.parquet')['Body'].read()))
print(f'subs: {len(subs):,}  agg: {len(agg):,}')

## 3 · Build area × month panel

BSS aggregated to (area, month). Join OSS aggregates to form panel data.

In [ ]:
bss_p = subs.groupby(['area','month_year']).agg(
    cem_target=('cem_score_target','mean'),
    rat_gap=('rat_gap_score','mean'),
    churn=('churn_risk_flag','mean'),
).reset_index()
oss_p = agg.groupby(['area','month_year']).agg(
    integ=('avg_integrity','mean'),
    cdr=('avg_cdr','mean'),
    tput=('avg_throughput_mbps','mean'),
    cells=('cell_count','sum'),
).reset_index()
panel = oss_p.merge(bss_p, on=['area','month_year']).sort_values(['area','month_year'])
print(f'panel: {len(panel):,} (area × month)')
panel.head()

## 4 · Granger tests per (cause → effect) pair

Test lags 1..4 months. Take min p-value across lags. Significant: `p < 0.05`.

In [ ]:
causes = GRANGER_CAUSES
effects = GRANGER_EFFECTS
results = []
for cause in causes:
    for effect in effects:
        pvals = []
        for area, g in panel.groupby('area'):
            if len(g) < MIN_PANEL_OBS: continue
            ts = g[[effect, cause]].dropna()
            if len(ts) < MIN_PANEL_OBS: continue
            try:
                test = grangercausalitytests(ts, maxlag=min(GRANGER_MAX_LAG, len(ts)-2), verbose=False)
                p = min(test[k][0][GRANGER_FTEST][1] for k in test)
                pvals.append(p)
            except Exception:
                continue
        if pvals:
            mp = float(np.median(pvals))
            results.append({'cause':cause, 'effect':effect,
                           'median_p':round(mp,4),
                           'significant':mp<GRANGER_ALPHA,
                           'areas_tested':len(pvals)})
res_df = pd.DataFrame(results)
print(res_df)

## 5 · Heatmap p-values

In [ ]:
pv = res_df.pivot(index='cause', columns='effect', values='median_p')
fig, ax = plt.subplots(figsize=(7,4))
sns.heatmap(pv, annot=True, fmt='.3f', cmap='RdYlGn_r', vmin=0, vmax=0.2, ax=ax,
            cbar_kws={'label':'median p-value'})
ax.set_title('Granger — OSS cause → BSS effect (lag ≤ 4)')
plt.tight_layout(); plt.show()

## 6 · Write granger_feature_gate.json

Offline gate consumed by api-gateway's `/granger-causality/lead-time` endpoint.

In [ ]:
gate = {
    'generated_at': pd.Timestamp.utcnow().isoformat(),
    'lag_max_months': GRANGER_MAX_LAG,
    'significance_threshold': GRANGER_ALPHA,
    'edges': res_df.to_dict(orient='records'),
    'significant_edges': res_df[res_df['significant']].to_dict(orient='records'),
}
out = Path(GATE_FNAME)
out.write_text(json.dumps(gate, indent=2))
print('Written', out)
print(json.dumps(gate, indent=2, default=str))
s3.put_object(Bucket='curated', Key=GATE_FNAME, Body=out.read_bytes())
print('↑ curated/granger_feature_gate.json')